# 📈 LSTM-Based Stock Price Sequence Prediction
### Lab Assignment 5 — AI Agent for Sequence Prediction

---

**Dataset:** Yahoo Finance API via `yfinance`  
**Stock:** RELIANCE.NS (Reliance Industries — NSE India)  
**Task:** Predict next-day closing price using past 60 days of data  
**Model:** LSTM (Long Short-Term Memory) Neural Network  
**Deployment:** FastAPI `/predict` endpoint + n8n WhatsApp Agent

---

### 🧮 LSTM Mathematical Model

An LSTM cell processes sequences using four gates:

| Gate | Formula | Purpose |
|------|---------|----------|
| **Forget Gate** | $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ | Decides what to forget from cell state |
| **Input Gate** | $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ | Decides what new info to store |
| **Cell Candidate** | $\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$ | New candidate values |
| **Cell State** | $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$ | Updated cell memory |
| **Output Gate** | $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ | Decides what to output |
| **Hidden State** | $h_t = o_t * \tanh(C_t)$ | Final output of LSTM cell |

Where $\sigma$ = sigmoid, $*$ = element-wise multiplication, $W$ = weight matrices, $b$ = bias vectors

---

## 📦 Section 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install yfinance --quiet
!pip install tensorflow --quiet
!pip install scikit-learn --quiet
!pip install plotly --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import yfinance as yf
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print("✅ All libraries imported successfully!")

## 📊 Section 2: Dataset Collection

**Dataset:** Yahoo Finance API (`yfinance`)  
**Source:** https://finance.yahoo.com  
**Stock:** RELIANCE.NS — Reliance Industries Limited (NSE India)  
**Features Used:** Open, High, Low, Close, Volume  
**Date Range:** 2018-01-01 to today

In [ ]:
# ─── Configuration ────────────────────────────────────────
TICKER      = 'RELIANCE.NS'   # NSE India stock
START_DATE  = '2018-01-01'
END_DATE    = pd.Timestamp.today().strftime('%Y-%m-%d')
SEQUENCE_LEN = 60             # Use past 60 days to predict next day
FEATURES    = ['Open', 'High', 'Low', 'Close', 'Volume']
TARGET_COL  = 'Close'         # We predict the closing price
TARGET_IDX  = FEATURES.index(TARGET_COL)  # index = 3

# ─── Download Data ─────────────────────────────────────────
print(f"📥 Downloading {TICKER} data from {START_DATE} to {END_DATE}...")
raw = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)

# Flatten multi-level columns if present
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[FEATURES].dropna().copy()

print(f"✅ Downloaded {len(df)} trading days")
print(f"   Date range: {df.index[0].date()} → {df.index[-1].date()}")
print(f"\n📋 Dataset shape: {df.shape}")
df.tail()

In [ ]:
# ─── Basic Statistics ──────────────────────────────────────
print("📊 Dataset Statistics:")
print(df.describe().round(2))
print(f"\n🔍 Missing values: {df.isnull().sum().sum()}")

In [ ]:
# ─── Visualize Raw Closing Price ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Closing price
axes[0].plot(df.index, df['Close'], color='royalblue', linewidth=1.2)
axes[0].set_title(f'{TICKER} — Closing Price (2018–Present)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (INR ₹)')
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(df.index, df['Close'], alpha=0.1, color='royalblue')

# Volume
axes[1].bar(df.index, df['Volume'], color='orange', alpha=0.6, width=1)
axes[1].set_title(f'{TICKER} — Trading Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('raw_data.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Chart saved as raw_data.png")

## 🔧 Section 3: Data Preprocessing

Steps:
1. **Normalize** all features to [0, 1] using MinMaxScaler
2. **Create sequences** — for each day *t*, use days *t-60* to *t-1* as input, day *t* as target
3. **Split** into 80% train, 20% test (time-aware — no shuffling)
4. **Reshape** to `(samples, timesteps, features)` for LSTM

In [ ]:
# ─── Step 3.1: Normalize ──────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[FEATURES].values)

print(f"✅ Data normalized to [0, 1]")
print(f"   Original Close range: ₹{df['Close'].min():.2f} — ₹{df['Close'].max():.2f}")
print(f"   Scaled Close range:   {scaled_data[:, TARGET_IDX].min():.3f} — {scaled_data[:, TARGET_IDX].max():.3f}")

In [ ]:
# ─── Step 3.2: Create Sequences ──────────────────────────
def create_sequences(data, seq_len, target_idx):
    """
    Build (X, y) pairs:
      X[i] = data[i : i+seq_len]          shape (seq_len, n_features)
      y[i] = data[i+seq_len, target_idx]  scalar — next day's Close
    """
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])          # window of 60 days
        y.append(data[i + seq_len, target_idx])  # next day close
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, SEQUENCE_LEN, TARGET_IDX)

print(f"✅ Sequences created:")
print(f"   X shape: {X.shape}  → (samples, timesteps, features)")
print(f"   y shape: {y.shape}  → (samples,)")

In [ ]:
# ─── Step 3.3: Train/Test Split (80/20, time-ordered) ────
split = int(len(X) * 0.80)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"✅ Train/Test Split:")
print(f"   Train: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Test:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"   Input shape per sample: {X_train.shape[1:]} → (60 days, 5 features)")

In [ ]:
# ─── Visualize a sample sequence ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(SEQUENCE_LEN), X_train[0, :, TARGET_IDX], 'b-o', markersize=3, label='Input Sequence (60 days)')
ax.axvline(x=SEQUENCE_LEN-1, color='red', linestyle='--', label='Prediction Point')
ax.scatter(SEQUENCE_LEN, y_train[0], color='red', s=100, zorder=5, label=f'Target (day 61)')
ax.set_title('Sample Training Sequence — Scaled Close Price', fontsize=13, fontweight='bold')
ax.set_xlabel('Timestep (days)')
ax.set_ylabel('Normalized Price')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("✅ Sequence visualization complete")

## 🧠 Section 4: LSTM Model Architecture

```
Input (60, 5)
    │
    ▼
LSTM Layer 1 — 128 units, return_sequences=True
    │  ← each timestep passes to next layer
Dropout (0.2)
    │
    ▼
LSTM Layer 2 — 64 units, return_sequences=True
    │
Dropout (0.2)
    │
    ▼
LSTM Layer 3 — 32 units, return_sequences=False
    │  ← only final timestep output
Dropout (0.2)
    │
    ▼
Dense (16, relu)
    │
    ▼
Dense (1)  ← predicted next-day Close
```

In [ ]:
# ─── Build LSTM Model ────────────────────────────────────
def build_lstm_model(seq_len, n_features):
    model = Sequential([
        # Layer 1 — LSTM with sequence output
        LSTM(128, return_sequences=True,
             input_shape=(seq_len, n_features),
             name='lstm_1'),
        Dropout(0.2, name='dropout_1'),

        # Layer 2 — LSTM with sequence output
        LSTM(64, return_sequences=True, name='lstm_2'),
        Dropout(0.2, name='dropout_2'),

        # Layer 3 — LSTM, only final output
        LSTM(32, return_sequences=False, name='lstm_3'),
        Dropout(0.2, name='dropout_3'),

        # Fully connected head
        Dense(16, activation='relu', name='dense_1'),
        Dense(1, name='output')  # scalar prediction
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mean_squared_error',
        metrics=['mae']
    )
    return model


model = build_lstm_model(SEQUENCE_LEN, len(FEATURES))
model.summary()

## 🏋️ Section 5: Model Training

In [ ]:
# ─── Callbacks ───────────────────────────────────────────
os.makedirs('model', exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'model/best_lstm.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

# ─── Train ────────────────────────────────────────────────
print("🚀 Starting training...")
print(f"   Training samples : {X_train.shape[0]}")
print(f"   Validation split : 10%")
print(f"   Batch size       : 32")
print(f"   Max epochs       : 100  (EarlyStopping active)")
print("-" * 50)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.10,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

In [ ]:
# ─── Training Curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', color='royalblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='orange')
axes[0].set_title('Model Loss (MSE)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE', color='green')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='red')
axes[1].set_title('Model MAE', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Training curves saved")

## 📉 Section 6: Prediction & Evaluation

In [ ]:
# ─── Generate Predictions ─────────────────────────────────
print("🔮 Generating predictions on test set...")
y_pred_scaled = model.predict(X_test, verbose=0)

# ─── Inverse Transform (back to ₹ prices) ────────────────
def inverse_close(scaled_vals):
    """Inverse-transform only the Close column (index 3)."""
    dummy = np.zeros((len(scaled_vals), len(FEATURES)))
    dummy[:, TARGET_IDX] = scaled_vals.flatten()
    return scaler.inverse_transform(dummy)[:, TARGET_IDX]

y_pred_actual = inverse_close(y_pred_scaled)
y_test_actual = inverse_close(y_test.reshape(-1, 1))

# ─── Metrics ──────────────────────────────────────────────
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae  = mean_absolute_error(y_test_actual, y_pred_actual)
mape = np.mean(np.abs((y_test_actual - y_pred_actual) / y_test_actual)) * 100
r2   = 1 - np.sum((y_test_actual - y_pred_actual)**2) / np.sum((y_test_actual - np.mean(y_test_actual))**2)

print("\n📊 Model Performance on Test Set:")
print(f"   RMSE  : ₹{rmse:.2f}")
print(f"   MAE   : ₹{mae:.2f}")
print(f"   MAPE  : {mape:.2f}%")
print(f"   R²    : {r2:.4f}")

In [ ]:
# ─── Plot: Actual vs Predicted ────────────────────────────
test_dates = df.index[split + SEQUENCE_LEN:]

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(test_dates, y_test_actual, label='Actual Close', color='royalblue', linewidth=1.5)
ax.plot(test_dates, y_pred_actual, label='Predicted Close', color='orange',
        linewidth=1.5, linestyle='--')

ax.set_title(f'{TICKER} — Actual vs Predicted Closing Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (INR ₹)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate metrics
textstr = f'RMSE: ₹{rmse:.2f}\nMAE: ₹{mae:.2f}\nMAPE: {mape:.2f}%\nR²: {r2:.4f}'
props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
ax.text(0.02, 0.95, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Actual vs Predicted chart saved")

In [ ]:
# ─── Scatter Plot: Actual vs Predicted ───────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test_actual, y_pred_actual, alpha=0.4, color='royalblue', s=15)
min_v = min(y_test_actual.min(), y_pred_actual.min())
max_v = max(y_test_actual.max(), y_pred_actual.max())
ax.plot([min_v, max_v], [min_v, max_v], 'r--', label='Perfect Prediction')
ax.set_xlabel('Actual Price (₹)', fontsize=12)
ax.set_ylabel('Predicted Price (₹)', fontsize=12)
ax.set_title('Scatter: Actual vs Predicted', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('scatter_actual_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔮 Section 7: Future Forecast (Next 7 Days)

In [ ]:
# ─── Multi-Step Future Prediction ────────────────────────
def forecast_future(model, scaler, df, features, target_idx, seq_len, n_days=7):
    """
    Iteratively predict n_days into the future.
    Each prediction is fed back as input for the next step.
    """
    last_window = df[features].values[-seq_len:]  # last 60 days
    temp = scaler.transform(last_window).copy()    # scale

    predictions = []
    for _ in range(n_days):
        x_input = temp.reshape(1, seq_len, len(features))
        pred_scaled = model.predict(x_input, verbose=0)[0, 0]

        # Build next row: copy last row, update Close column
        new_row = temp[-1].copy()
        new_row[target_idx] = pred_scaled

        # Inverse transform to get ₹ price
        dummy = np.zeros((1, len(features)))
        dummy[0] = new_row
        price = scaler.inverse_transform(dummy)[0, target_idx]
        predictions.append(price)

        # Slide window forward
        temp = np.vstack([temp[1:], new_row])

    return predictions


future_prices = forecast_future(model, scaler, df, FEATURES, TARGET_IDX, SEQUENCE_LEN, n_days=7)

# Build future dates (skip weekends simply)
last_date = df.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=7)

future_df = pd.DataFrame({'Date': future_dates, 'Predicted_Close': future_prices})
future_df['Date'] = future_df['Date'].dt.strftime('%d-%b-%Y')

current_price = df['Close'].iloc[-1]

print(f"\n📅 7-Day Forecast for {TICKER}")
print(f"   Last Known Price: ₹{current_price:.2f}")
print("-" * 35)
for _, row in future_df.iterrows():
    change = ((row['Predicted_Close'] - current_price) / current_price) * 100
    arrow = "▲" if change > 0 else "▼"
    print(f"   {row['Date']}  →  ₹{row['Predicted_Close']:,.2f}  {arrow} {change:+.2f}%")

avg_pred = np.mean(future_prices)
overall_change = ((avg_pred - current_price) / current_price) * 100
print(f"\n📊 Avg Forecast: ₹{avg_pred:,.2f}  ({overall_change:+.2f}% expected change)")

if overall_change > 2:
    print("🟢 Signal: STRONG BUY")
elif overall_change > 0.5:
    print("🟡 Signal: BUY / ACCUMULATE")
elif overall_change > -0.5:
    print("🟠 Signal: HOLD")
else:
    print("🔴 Signal: SELL")

In [ ]:
# ─── Forecast Chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

# Last 60 days actual
recent = df['Close'].tail(60)
ax.plot(recent.index, recent.values, color='royalblue', linewidth=1.5, label='Historical Close')

# Forecast
ax.plot(future_dates, future_prices, 'o--', color='orange',
        linewidth=2, markersize=7, label='7-Day Forecast')

# Connecting line
ax.plot([recent.index[-1], future_dates[0]],
        [recent.values[-1], future_prices[0]], '--', color='gray', alpha=0.6)

ax.axvline(x=recent.index[-1], color='gray', linestyle=':', alpha=0.7)
ax.set_title(f'{TICKER} — 7-Day Price Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (INR ₹)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_7day.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ 7-day forecast chart saved")

## 💾 Section 8: Save Model & Scaler

In [ ]:
# ─── Save Artifacts ──────────────────────────────────────
os.makedirs('model', exist_ok=True)

# Save final model
model.save('model/lstm_model.h5')
print("✅ Model saved → model/lstm_model.h5")

# Save scaler
joblib.dump(scaler, 'model/scaler.pkl')
print("✅ Scaler saved → model/scaler.pkl")

# Verify sizes
model_size = os.path.getsize('model/lstm_model.h5') / 1024
scaler_size = os.path.getsize('model/scaler.pkl') / 1024
print(f"\n   lstm_model.h5 : {model_size:.1f} KB")
print(f"   scaler.pkl    : {scaler_size:.1f} KB")

In [ ]:
# ─── Download model files from Colab ─────────────────────
try:
    from google.colab import files
    print("📥 Downloading model files to your computer...")
    files.download('model/lstm_model.h5')
    files.download('model/scaler.pkl')
    print("✅ Files downloaded!")
except ImportError:
    print("ℹ️  Not running in Colab — files are saved in ./model/ directory")

## ✅ Section 9: Summary

### What We Built

| Component | Details |
|-----------|--------|
| **Dataset** | RELIANCE.NS stock data via Yahoo Finance API |
| **Features** | Open, High, Low, Close, Volume (5 features) |
| **Sequence Length** | 60 trading days |
| **Model** | 3-layer Stacked LSTM + Dense head |
| **Parameters** | ~110,000 trainable parameters |
| **Loss Function** | Mean Squared Error (MSE) |
| **Optimizer** | Adam (lr=0.001) |
| **Output** | Next-day closing price prediction |

### LSTM Gate Summary (for presentation)

- **Forget Gate** ($f_t$): Reads $h_{t-1}$ and $x_t$, outputs 0–1 per cell state dimension. 0 = forget, 1 = keep.
- **Input Gate** ($i_t$): Decides how much of the new candidate $\tilde{C}_t$ to write into the cell state.
- **Cell State** ($C_t$): Long-term memory. Updated by forgetting old + adding new information.
- **Output Gate** ($o_t$): Controls what portion of the cell state becomes the hidden state $h_t$.
- **Hidden State** ($h_t$): Short-term memory / current output, passed to the next timestep.

### Deployment
- **FastAPI** serves `/predict?symbol=RELIANCE` → returns JSON with predicted price + BUY/SELL signal
- **n8n** automation polls FastAPI every 5 minutes → sends WhatsApp alert on BUY signals via Twilio
- **Streamlit** dashboard provides real-time visualization

---
*AI Tools Acknowledgement: Claude (Anthropic) was used to assist in code structuring and documentation.*